# Capa silver: accidentes en carreteras

Limpieza y tipado de bronze. Se mantiene una sola tabla.

- **Entrada:** `data/bronze/accidentes.parquet`
- **Salida:** `data/silver/accidentes.parquet`
- **Grano:** una fila por accidente
- **Reglas de limpieza:** hallazgos de `exploration/01_perfilado_bronze.ipynb`

## Configuración

In [134]:
import pandas as pd
from pathlib import Path

origen = Path("../data/bronze/accidentes.parquet")
destino = Path("../data/silver/accidentes.parquet")

pd.set_option("display.max_columns", None)

## Lectura

Esperado: 60.531 filas, 22 columnas.

In [135]:
df = pd.read_parquet(origen)
filas_bronze = len(df)
df.shape

(60531, 22)

## Renombrado de columnas

Convención: snake_case, en español y sin acentos.

In [136]:
df = df.rename(columns={
    "Fecha": "fecha",
    "uf": "estado",
    "tipo_acidente": "tipo_accidente",
    "clasificación_accidente": "clasificacion_accidente",
    "Personas": "personas",
    "Muertos": "muertos",
    "Herido_Leves": "heridos_leves",
    "Heridos_graves": "heridos_graves",
    "Heridos": "heridos",
    "vehículos": "vehiculos",
    "latitude": "latitud",
    "longitude": "longitud",
    "tipo_pista": "tipo_carril"
})
df.columns.to_list()

['id',
 'fecha',
 'hora',
 'estado',
 'municipio',
 'causa_accidente',
 'tipo_accidente',
 'clasificacion_accidente',
 'sentido_via',
 'tipo_carril',
 'personas',
 'muertos',
 'heridos_leves',
 'heridos_graves',
 'ilesos',
 'ignorados',
 'heridos',
 'vehiculos',
 'latitud',
 'longitud',
 '_fecha_carga',
 '_archivo_origen']

## Tipos de datos

In [137]:
# Convierte a tipo date
df["fecha"] = df["fecha"].dt.date

`hora` llega como tipo time y no requiere conversión.

## Reemplazar texto de categorías

In [138]:
# Columna: causa_accidente
causas = {
    "Condutor Dormindo": "Conductor dormido",
    "Transitar no acostamento": "Transitar por el costado de la vía",
    "Objeto estático en la plataforma del vagón": "Objeto estático sobre la vía",
    "No mantengas una distancia segura.": "No mantuvo una distancia segura"
}

df["causa_accidente"] = df["causa_accidente"].replace(causas)

df["causa_accidente"] = df["causa_accidente"].str.rstrip(".")

df["causa_accidente"] = df["causa_accidente"].str.replace("Fenomenos", "Fenómenos")

In [139]:
# Columna: sentido_via
df["sentido_via"] = df["sentido_via"].replace("DeCreciente", "Decreciente")

In [140]:
# Columna: tipo_accidente
tipos_accidentes = {
    "Queda de ocupante de veículo": "Caída de un ocupante de un vehículo",
    "Salida de la cama del carro": "Salida de la vía",
    "Listado": "Volcamiento lateral",
    "Peatón atropellando": "Atropello de peatón",
    "Posibles daños": "Daños eventuales",
    "Estancamiento": "Colisión múltiple",
    "Desconexión de carga": "Derrame de carga"
}
df["tipo_accidente"] = df["tipo_accidente"].replace(tipos_accidentes)

In [141]:
# Columna: tipo_carril
df["tipo_carril"] = df["tipo_carril"].replace("Dupla", "Doble")

## Unificar categorías

In [142]:
# Columna: causa_accidente
causas_unificadas = {
    "Consumo de alcohol": "Consumo de alcohol por parte del conductor",
    "Enfermedad repentina": "Enfermedad repentina del conductor"
}

df["causa_accidente"] = df["causa_accidente"].replace(causas_unificadas)

## Validaciones

In [143]:
assert len(df) == filas_bronze, "silver no tiene las mismas filas que bronze"
assert df["id"].is_unique, "hay id duplicados"
assert df.isna().sum().sum() == 0, "hay valores nulos"
assert (df["heridos"] == df["heridos_leves"] + df["heridos_graves"]).all(), "heridos no coincide con leves + graves"
assert (df["personas"] == df["muertos"] + df["heridos"] + df["ilesos"] + df["ignorados"]).all(), "personas no coincide con la suma de sus partes"

originales = list(causas) + list(causas_unificadas) + list(tipos_accidentes)
assert not df[["causa_accidente", "tipo_accidente"]].isin(originales).any().any(), "quedan valores sin reemplazar"

## Escritura

In [144]:
destino.parent.mkdir(exist_ok=True)
df.to_parquet(destino, index=False)

In [145]:
pd.read_parquet(destino).info()

<class 'pandas.DataFrame'>
RangeIndex: 60531 entries, 0 to 60530
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   id                       60531 non-null  int64         
 1   fecha                    60531 non-null  object        
 2   hora                     60531 non-null  object        
 3   estado                   60531 non-null  str           
 4   municipio                60531 non-null  str           
 5   causa_accidente          60531 non-null  str           
 6   tipo_accidente           60531 non-null  str           
 7   clasificacion_accidente  60531 non-null  str           
 8   sentido_via              60531 non-null  str           
 9   tipo_carril              60531 non-null  str           
 10  personas                 60531 non-null  int64         
 11  muertos                  60531 non-null  int64         
 12  heridos_leves            60531 non-null  in